# Baseline: previsão de sobrevivência no Titanic
**Problema:** classificação binária (sobreviveu = 1 / não = 0).
**Meta:** criar um modelo baseline e avaliá-lo com precisão, recall e F1.

In [13]:
#imports e semente
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_score, recall_score)

SEED = 42 #semente aleatória para reprodutibilidade dos experimentos -> aleatoriedade para reproduzir os resultados
# A reprodutibilidade dos experimentos significa conseguir executar um experimento novamente e obter os mesmos resultados, mantendo as mesmas condições.

In [14]:
#carregar e inspecionar dados
df = sns.load_dataset("titanic")
df[["survived", "pclass", "sex", "age", "fare"]].head()

,survived,pclass,sex,age,fare
0,0,3,male,22.0,7.2500
1,1,1,female,38.0,71.2833
2,1,3,female,26.0,7.9250
3,1,1,female,35.0,53.1000
4,0,3,male,35.0,8.0500


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [16]:
df[["survived", "pclass", "sex", "age", "fare"]].isna().sum()

survived      0
pclass        0
sex           0
age         177
fare          0
dtype: int64

In [17]:
df["survived"].value_counts(normalize=True)

survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

* age tem valores nulos
* só 38% sobrevivem, então um modelo que sempre responde "não sobreviveram" já acertaria 62%

In [18]:
#seleciona colunas e trasnforma variável de texto em número -> modelos só aceitam números
df = df[["survived", "pclass", "sex", "age", "fare"]].copy()
df["sex"] = (df["sex"] == "female").astype(int)

X = df.drop(columns="survived")   # features
y = df["survived"]                # target

In [19]:
# separa dados de treino e testes
X_tr, X_te, y_tr, y_te = train_test_split( #split
    X, y, test_size=0.2, stratify=y, random_state=SEED) # separa arrays e matrizes em subconjuntos aleatórios de treinos e testes
#test_size=0.2 -> porcentagem de dados reservados para teste
#stratify=y ->garante que a proporção de classes da variável target seja mantida nos conjuntos de treino e teste
#random_state=42 -> garante que a divisão aleatória dos dados seja reproduzível, desde que sejam mantidos os mesmos dados, parâmetros e condições de execução. 
# -> permite repetir exatamente a mesma aleatoriedade

In [20]:
#tratar nulos usando treino
mediana_idade = X_tr["age"].median()
X_tr["age"] = X_tr["age"].fillna(mediana_idade)
X_te["age"] = X_te["age"].fillna(mediana_idade)

# a mediana é calculada só com o treino e depois aplicada ao teste. Se você calculasse com todos os dados, 
# informação do teste vazaria para o treino (data leakage), e a avaliação ficaria otimista demais. Na primeira versão do código que passei, 
# o preenchimento vinha antes do split; esta ordem é a correta, e vale citá-la na entrevista.

In [21]:
#baseline burro
dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print("Acurácia do baseline burro:", round(dummy.score(X_te, y_te), 3))

Acurácia do baseline burro: 0.615


Por quê: no trabalho, todo modelo é comparado com uma referência simples. Se o seu modelo não bate o burro, não serve.

In [ ]:
#treinar o modelo de regressão logística
modelo = LogisticRegression(max_iter=1000).fit(X_tr, y_tr) #criação do modelo

print("Treino:", round(modelo.score(X_tr, y_tr), 3)) #score calcula a acurácia do modelo (proporção de acertos sobre o total de previsões)
print("Teste: ", round(modelo.score(X_te, y_te), 3))

Treino: 0.801
Teste:  0.788


In [23]:
pred = modelo.predict(X_te) #usa modelo treinado para realizar previsões dos dados do conjunto de testes
print(confusion_matrix(y_te, pred)) #matriz de confusção -> mostra a quantidade de erros e acertos do modelo
print(classification_report(y_te, pred))#gera relatório de métricas de classificação

[[94 16]
 [22 47]]
              precision    recall  f1-score   support

           0       0.81      0.85      0.83       110
           1       0.75      0.68      0.71        69

    accuracy                           0.79       179
   macro avg       0.78      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179



In [24]:
#validação cruzada
scores = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring="f1")
print(scores.round(3), "| média:", scores.mean().round(3))

[0.708 0.617 0.748 0.762 0.741] | média: 0.715


* modelo: modelo de Machine Learning que será avaliado.
* X_tr, y_tr: dados de treinamento.
* cv=5: divide os dados em 5 partes (folds). O modelo será treinado e avaliado 5 vezes, utilizando uma parte diferente para validação em cada execução.
* scoring="f1": utiliza a métrica F1-score para avaliar o desempenho em cada divisão.

In [25]:
proba = modelo.predict_proba(X_te)[:, 1]
for t in [0.2, 0.35, 0.5, 0.7]:
    p = (proba >= t).astype(int)
    print(t, "precisão:", round(precision_score(y_te, p), 2),
             "recall:", round(recall_score(y_te, p), 2))

0.2 precisão: 0.55 recall: 0.84
0.35 precisão: 0.67 recall: 0.8
0.5 precisão: 0.75 recall: 0.68
0.7 precisão: 0.89 recall: 0.45
